<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: كورجون ديمتري، @tbb
    
## <center> البرنامج التعليمي
### <center> "شيء آخر حول التعلم الجماعي"



الهدف من أساليب التجميع هو الجمع بين المصنفات المختلفة في مصنف تعريفي يتمتع بأداء تعميمي أفضل من كل مصنف فردي على حدة. على سبيل المثال، بافتراض أننا جمعنا تنبؤات من 10 نواة مختلفة، فإن طريقة التجميع ستسمح لنا بدمج هذه التنبؤات للتوصل إلى تنبؤ أكثر دقة وقوة من التنبؤ بنواة واحدة. هناك عدة طرق لإنشاء مجموعة من المصنفات تهدف كل منها إلى غرض خاص:
* **التعبئة** - تقليل التباين
* **التعزيز** - تقليل التحيز
* **التراص** - تحسين القوة التنبؤية
ما هو "التعبئة" و"التعزيز" الذي تعرفه بالفعل من المحاضرات، لكن دعني أذكرك بالأفكار الرئيسية.
**_Bagger_** - إنشاء بيانات إضافية للتدريب من مجموعة البيانات الأصلية باستخدام مجموعات مع التكرار لإنتاج مجموعات متعددة بنفس حجم مجموعة البيانات الأصلية. من خلال زيادة حجم مجموعة التدريب، لا يمكنك تحسين القوة التنبؤية للنموذج، ولكن يمكنك فقط تقليل التباين، وضبط التنبؤ بشكل ضيق على النتيجة المتوقعة.**_Boosting_** - نهج من خطوتين، حيث يستخدم أولاً مجموعات فرعية من البيانات الأصلية لإنتاج سلسلة من النماذج ذات الأداء المتوسط ​​ثم "يعزز" أدائها من خلال دمجها معًا باستخدام دالة تكلفة معينة (على سبيل المثال، تصويت الأغلبية). على عكس التعبئة، في التعزيز الكلاسيكي، لا يكون إنشاء المجموعة الفرعية عشوائيًا ويعتمد على أداء النماذج السابقة: تحتوي كل مجموعة فرعية جديدة على العناصر التي تم تصنيفها بشكل خاطئ بواسطة النموذج السابق.
**_التكديس (المزج)_ ** - يشبه التعزيز: يمكنك أيضًا تطبيق عدة نماذج على بياناتك الأصلية. الفرق هنا هو أنه ليس لديك صيغة تجريبية لوظيفة الوزن الخاصة بك، بل يمكنك تقديم مستوى تعريفي واستخدام نموذج/نهج آخر لتقدير المدخلات مع مخرجات كل نموذج لتقدير الأوزان، وبعبارة أخرى، لتحديد النماذج التي تؤدي أداءً جيدًا والنماذج التي تعطي بيانات الإدخال هذه بشكل سيء.
### مقدمة
قبل أن نبدأ، أعتقد أننا يجب أن نرى رسمًا بيانيًا يوضح العلاقة بين خطأ المجموعة والمصنف الفردي. وبعبارة أخرى، يصور هذا الرسم البياني نظرية هيئة المحلفين في كوندورسيه.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

from itertools import product

from scipy.misc import comb

In [ ]:
# calculate ensemble error
def ensemble_error(n_clf, error):
    k_start = math.ceil(n_clf / 2)
    probs = [
        comb(n_clf, k) * error ** k * (1 - error) ** (n_clf - k)
        for k in range(k_start, n_clf + 1)
    ]
    return sum(probs)


error_range = np.arange(0.0, 1.01, 0.01)
errors = [ensemble_error(n_clf=11, error=error) for error in error_range]

plt.plot(error_range, errors, label="Ensemble error", linewidth=2)
plt.plot(error_range, error_range, linestyle="--", label="Base error", linewidth=2)
plt.xlabel("Base error")
plt.ylabel("Base/Ensemble error")
plt.legend(loc="best")
plt.grid()
plt.show()


كما نرى، فإن احتمالية الخطأ للمجموعة تكون دائمًا أفضل من خطأ المصنف الفردي طالما أن أداء المصنف أفضل من التخمين العشوائي.
لنبدأ بتمرين إحماء ونطبق مصنفًا مجمعًا بسيطًا لتصويت الأغلبية كمثال لأبسط خوارزمية المجموعة.


In [ ]:
import warnings

from sklearn import datasets
# import some useful stuff
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import (GridSearchCV, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, _name_estimators
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

In [ ]:
# and make a small helper function to plot classifiers decision area
def plot_clf_area(
    classifiers, labels, X, s_row=2, s_col=2, scaling=True, colors=None, markers=None
):
    if not colors:
        colors = ["green", "red", "blue"]

    if not markers:
        markers = ["^", "o", "x"]

    if scaling:
        sc = StandardScaler()
        X_std = sc.fit_transform(X)

    # find plot boundaries
    x_min = X_std[:, 0].min() - 1
    x_max = X_std[:, 0].max() + 1
    y_min = X_std[:, 1].min() - 1
    y_max = X_std[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

    f, axarr = plt.subplots(
        nrows=s_row, ncols=s_col, sharex="col", sharey="row", figsize=(12, 8)
    )
    for idx, clf, tt in zip(product(range(s_row), range(s_col)), classifiers, labels):
        clf.fit(X_std, y_train)
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        axarr[idx[0], idx[1]].contourf(xx, yy, Z, alpha=0.3)

        for label, color, marker in zip(np.unique(y_train), colors, markers):
            axarr[idx[0], idx[1]].scatter(
                X_std[y_train == label, 0],
                X_std[y_train == label, 1],
                c=color,
                marker=marker,
                s=50,
            )
        axarr[idx[0], idx[1]].set_title(tt)


### تنفيذ مُصنف أصوات الأغلبية البسيطة


In [ ]:
class MajorityVoteClassifier(BaseEstimator, ClassifierMixin):
    """
    A Majority vote ensemble classifier
    
    Params
    -----
    classifiers : array, shape = [n_classifiers]
        Classifiers for the ensemble
      
    vote : str, {'label', 'probability'}
        Default: 'label'
        If 'label' the prediction based on the argmax
        of class labels. Else if 'probability', the
        argmax of the sum of probabilities is used to
        predict the class label.
      
    weights : array, shape = [n_classifiers]
        Optional, default: None
        If a list of 'int' or 'float' values are provided,
        the classifiers are weighted by importance;
        Uses uniform weights if 'None'
    """

    def __init__(self, classifiers, vote="label", weights=None):
        self.classifiers = classifiers
        self.named_classifiers = {
            key: value for key, value in _name_estimators(classifiers)
        }
        self.vote = vote
        self.weights = weights

    def fit(self, X, y):
        """
        Fit classifiers.
        
        Params
        -----
        X : {array, matrix}
            shape = [n_samples, n_features]
            Matrix of training samples.
            
        y : array, shape = [n_samples]
            Vector of target labels.
        """

        # Use LabelEncoder to ensure class labels start with 0
        # which is important for np.argmax call in self.predict
        self.le_ = LabelEncoder()
        self.le_.fit(y)
        self.classes_ = self.le_.classes_
        self.classifiers_ = []
        for clf in self.classifiers:
            fitted_clf = clone(clf).fit(X, self.le_.transform(y))
            self.classifiers_.append(fitted_clf)
        return self

    def predict(self, X):
        """
        Predict class labels for X.
        
        Params
        -----
        X : {array, matrix}
            shape = [n_samples, n_features]
            Matrix of training samples.
            
        Returns
        -----
        maj_vote : array, shape = [n_samples]
            Predicted class labels.
        """

        if self.vote == "probability":
            maj_vote = np.argmax(self.predict_proba(X), axis=1)
        else:
            predictions = np.asarray([clf.predict(X) for clf in self.classifiers_]).T
            maj_vote = np.apply_along_axis(
                lambda x: np.argmax(np.bincount(x, weights=self.weights)),
                axis=1,
                arr=predictions,
            )

        maj_vote = self.le_.inverse_transform(maj_vote)
        return maj_vote

    def predict_proba(self, X):
        """
        Predict class probabilities for X.

        Params
        -----
        X : {array, matrix}
            shape = [n_samples, n_features]
            Training vectors, where n_samples is the number
            of samples and n_features the number of features.

        Returns
        -----
        avg_proba : array
            shape = [n_samples, n_classes]
            Weighted average probability for each class per sample
        """
        probas = np.asarray([clf.predict_proba(X) for clf in self.classifiers_])
        avg_proba = np.average(probas, axis=0, weights=self.weights)
        return avg_proba


توفر الفئات الأصلية **_BaseEstimator_** و **_ClassifierMixin_** بعض الوظائف الأساسية مثل *get_params* و *set_params* مجانًا.
الآن حان الوقت لاختبار المصنف.


In [ ]:
# load data
wine = datasets.load_wine()
wine.feature_names

In [ ]:
# use only two feature - alco & hue
X, y = wine.data[:, [0, 10]], wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=11
)

In [ ]:
# make base classifiers
clf1 = LogisticRegression(penalty="l2", C=0.001, random_state=11)
clf2 = DecisionTreeClassifier(max_depth=2, criterion="entropy", random_state=11)
clf3 = KNeighborsClassifier(n_neighbors=1, p=2, metric="minkowski")

# LR and KNN use Euclidian distance metric so need to scale the data
pipe1 = Pipeline([["sc", StandardScaler()], ["clf", clf1]])
pipe3 = Pipeline([["sc", StandardScaler()], ["clf", clf3]])

mv_clf = MajorityVoteClassifier(classifiers=[pipe1, clf2, pipe3])

labels = ["Logistic Regresion", "Decision Tree", "KNN", "Majority Vote"]

all_clf = [pipe1, clf2, pipe3, mv_clf]
for clf, label in zip(all_clf, labels):
    scores = cross_val_score(estimator=clf, X=X_train, y=y_train, cv=10)
    print(f"ROC AUC: {scores.mean():.2f} (+/- {scores.std():.2f} {label})")


plot_clf_area(all_clf, labels, X_train)
plt.show()

كما يمكننا أن نرى، تحسن أداء غاز MajorityVotingClassifier بشكل جزئي مقارنة بالمصنفات الفردية في تقييم التحقق من الصحة المتبادل بعشرة أضعاف. لاحظ أن مناطق القرار الخاصة بمصنف المجموعة تبدو وكأنها مزيج من مناطق القرار من المصنفات الفردية. 



### التراص
نهج تصويت الأغلبية يشبه التراص. ومع ذلك، يتم استخدام خوارزمية التراص مع نموذج يتنبأ بتسمية الفئة النهائية باستخدام تنبؤات المصنفات الفردية في المجموعة كمدخلات.
الفكرة الأساسية وراء التعميم المكدس هي استخدام مجموعة من المصنفات الأساسية، ثم استخدام مصنف آخر، يسمى المصنف الوصفي، لدمج توقعاتهم، بهدف تقليل خطأ التعميم.
لنفترض أنك تريد إجراء عملية تكديس ثنائية:
* قم بتقسيم مجموعة القطار إلى جزأين: Train_a وtrain_b
* قم بتركيب نموذج المرحلة الأولى على Train_a وقم بإنشاء تنبؤات لـ Train_b
* قم بتركيب نفس النموذج على Train_b وقم بإنشاء تنبؤات لـ Train_a
* أخيرًا قم بتركيب النموذج على مجموعة القطار بأكملها وقم بإنشاء تنبؤات لمجموعة الاختبار.
* الآن قم بتدريب نموذج مكدس المرحلة الثانية على الاحتمالات من نموذج (نماذج) المرحلة الأولى.
سوف نستخدم فقط الميزات التعريفية والتحقق من صحة الكتلة الواحدة. يمكنك بسهولة إضافة الوظائف الضرورية إذا كنت بحاجة.
دع تطبيق التراص يعتمد على فئة MajorityVoteClassifier. 


In [ ]:
class StackingClassifier(BaseEstimator, ClassifierMixin):
    """A Stacking classifier for scikit-learn estimators for classification.
    
    Params
    -----
    classifiers : array, shape = [n_classifiers]
        A list of classifiers for stacking.
    meta_classifier : object
        The meta-classifier to be fitted on the ensemble of
        classifiers
    use_probas : bool (default: True)
        If True, trains meta-classifier based on predicted probabilities
        instead of class labels.
    average_probas : bool (default: True)
        Averages the probabilities as meta features if True.

    """

    def __init__(
        self, classifiers, meta_classifier, use_probas=True, average_probas=True
    ):

        self.classifiers = classifiers
        self.meta_classifier = meta_classifier
        self.named_classifiers = {
            key: value for key, value in _name_estimators(classifiers)
        }
        self.named_meta_classifier = {
            f"meta-{key}": value for key, value in _name_estimators([meta_classifier])
        }
        self.use_probas = use_probas
        self.average_probas = average_probas

    def fit(self, X, y):
        """ Fit ensemble classifers and the meta-classifier.
        
        Params
        -----
        X : {array, matrix}, shape = [n_samples, n_features]
            Training vectors, where n_samples is the number of samples and
            n_features is the number of features.
        y : array, shape = [n_samples] or [n_samples, n_outputs]
            Target values.
        """
        self.classifiers_ = [clone(clf) for clf in self.classifiers]
        self.meta_clf_ = clone(self.meta_classifier)

        for clf in self.classifiers_:
            clf.fit(X, y)

        meta_features = self.predict_meta_features(X)
        self.meta_clf_.fit(meta_features, y)
        return self

    def predict(self, X):
        """ Predict target values for X.
        Params
        -----
        X : {array, matrix}, shape = [n_samples, n_features]
            Training vectors, where n_samples is the number of samples and
            n_features is the number of features.

        Returns
        -----
        labels : array, shape = [n_samples] or [n_samples, n_outputs]
            Predicted class labels.
        """
        meta_features = self.predict_meta_features(X)

        return self.meta_clf_.predict(meta_features)

    def predict_proba(self, X):
        """ Predict class probabilities for X.
        Params
        -----
        X : {array, matrix}, shape = [n_samples, n_features]
            Training vectors, where n_samples is the number of samples and
            n_features is the number of features.

        Returns
        -----
        proba : array, shape = [n_samples, n_classes] or a list of \
                n_outputs of such arrays if n_outputs > 1.
            Probability for each class per sample.
        """
        meta_features = self.predict_meta_features(X)

        return self.meta_clf_.predict_proba(meta_features)

    def predict_meta_features(self, X):
        """ Get meta-features of test-data.
        Params
        -----
        X : array, shape = [n_samples, n_features]
            Test vectors, where n_samples is the number of samples and
            n_features is the number of features.

        Returns
        -----
        meta-features : array, shape = [n_samples, n_classifiers]
            Returns the meta-features for test data.
        """
        if self.use_probas:
            probas = np.asarray([clf.predict_proba(X) for clf in self.classifiers_])
            if self.average_probas:
                vals = np.average(probas, axis=0)
            else:
                vals = np.concatenate(probas, axis=1)
        else:
            vals = np.column_stack([clf.predict(X) for clf in self.classifiers_])
        return vals

    def get_params(self, deep=True):
        """Return estimator parameter names for GridSearch support."""

        if not deep:
            return super(StackingClassifier, self).get_params(deep=False)
        else:
            out = self.named_classifiers.copy()
            for name, step in self.named_classifiers.items():
                for key, value in step.get_params(deep=True).items():
                    out[f"{name}__{key}"] = value

            out.update(self.named_meta_classifier.copy())
            for name, step in self.named_meta_classifier.items():
                for key, value in step.get_params(deep=True).items():
                    out[f"{name}__{key}"] = value

            for key, value in (
                super(StackingClassifier, self).get_params(deep=False).items()
            ):
                out[f"{key}"] = value

            return out


عادة، يتم استخدام **_LogisticRegression_** كنموذج تعريفي ولن نغير التقليد. دعونا نتحقق من StackingClassifier.


In [ ]:
# make base LR classifiers
lr1 = LogisticRegression(C=0.1, random_state=11)
lr2 = LogisticRegression(C=1, random_state=11)
lr3 = LogisticRegression(C=10, random_state=11)

# make base DT classifiers
dt1 = DecisionTreeClassifier(max_depth=1, random_state=11)
dt2 = DecisionTreeClassifier(max_depth=2, random_state=11)
dt3 = DecisionTreeClassifier(max_depth=3, random_state=11)

# make base KNN classifiers
knn1 = KNeighborsClassifier(n_neighbors=1)
knn2 = KNeighborsClassifier(n_neighbors=2)

# scale data for metrics classifiers
pipe1 = Pipeline([["sc", StandardScaler()], ["clf", lr1]])
pipe2 = Pipeline([["sc", StandardScaler()], ["clf", lr2]])
pipe3 = Pipeline([["sc", StandardScaler()], ["clf", lr3]])
pipe4 = Pipeline([["sc", StandardScaler()], ["clf", knn1]])
pipe5 = Pipeline([["sc", StandardScaler()], ["clf", knn2]])
clfs = [pipe1, pipe2, pipe3, dt1, dt2, dt3, pipe4, pipe5]

# make meta classifiers
meta_clf = LogisticRegression(random_state=11)
stacking = StackingClassifier(classifiers=clfs, meta_classifier=meta_clf)

labels = [
    "Logistic Regresion C=0.1",
    "Logistic Regresion C=1",
    "Logistic Regresion C=10",
    "Decision Tree depth=1",
    "Decision Tree depth=2",
    "Decision Tree depth=3",
    "KNN 1",
    "KNN 2",
    "Stacking",
]

clfs = clfs + [stacking]

for clf, label in zip(clfs, labels):
    scores = cross_val_score(estimator=clf, X=X_train, y=y_train, cv=10)
    print(f"ROC AUC: {scores.mean():.2f} (+/- {scores.std():.2f} {label})")

plot_clf_area(clfs, labels, X_train, s_row=3, s_col=3)
plt.show()


### المزج
"المزج" هي كلمة قدمها الفائزون في Netflix. إنه قريب جدًا من التعميم المكدس، ولكنه أبسط قليلاً وأقل خطرًا لتسرب المعلومات. يستخدم بعض الباحثين مصطلحي "التجميع المكدس" و"المزج" بالتبادل.باستخدام المزج، بدلاً من إنشاء تنبؤات خارج الطية لمجموعة القطار، يمكنك إنشاء مجموعة صغيرة من الإيقاف تمثل 10% من مجموعة القطار مثلاً. يتم بعد ذلك تدريب نموذج المكدس على مجموعة الإيقاف هذه فقط.
المزج له فوائد قليلة:
* إنه أبسط من التراص.
* يمنع تسرب المعلومات: يستخدم المعممون والمكدسون بيانات مختلفة.
السلبيات هي:
* أنت تستخدم بيانات أقل بشكل عام
* قد يتناسب النموذج النهائي مع مجموعة الإيقاف.



### ملخص
تجمع الأساليب المجمعة بين نماذج تصنيف مختلفة لإلغاء نقاط ضعفها الفردية، مما يؤدي غالبًا إلى نماذج مستقرة وجيدة الأداء تكون جذابة جدًا لمسابقات التعلم الآلي وأحيانًا للتطبيقات الصناعية أيضًا.



### الموارد
1. [التعلم الجماعي لتحسين نتائج التعلم الآلي](https://blog.statsbot.co/ensemble-learning-d1dcd548e936)
2. [دليل فرقة KAGGLE](https://mlwave.com/kaggle-ensembling-guide/)
3. [حل BigChaos لجائزة Netflix الكبرى](https://www.netflixprize.com/assets/GrandPrize2009_BPC_BigChaos.pdf)
4. [التجميع الخطي الموزون للميزات](https://arxiv.org/pdf/0911.0460.pdf)
5. [مثال التراص](https://github.com/Dyakonov/ml_hacks/blob/master/dj_stacking.ipynb)
6. [دليل Kaggler لتكديس النماذج عمليًا](http://blog.kaggle.com/2016/12/27/a-kagglers-guide-to-model-stacking-in-practice/)